**Purpose:** Synthetic Reddit posts for validation of the sentiment model.

**Inputs:** `reddit_synthetic_labeled_sy2194.parquet` (one-off labeled export, not in repo), `02_sentiment/reddit/dataset.parquet`

**Outputs:** `all_synthetic_labeled.parquet`, `reddit_synthetic_labeled_XXX.parquet`, `synthetic_posts.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


foram selecionados aleatoriamente 20 posts de cada setor, por cada sentimento, por nível de concordância (2 ou 3) (apenas para os setores possíveis, alguns não chegavam a 20 posts, então foram na totalidade, o máximo possível (por ex. Utilities)).

depois para esta amostra de 20 posts (60*2 por setor, quando possível), foi perguntado ao chatpgt 5.2 flagship (tal como na situação das notícias) qual o sentimento do post, e depois comparado com o sentimento esperado (pelas 3 llms) (resultados em inputs/load.ipynb)

onde ele concordou, será o que vai ser usado para teste final. o que sobra é para treino.

o facto de ter esgotados posts de alguns setores, faz com que tenha 0 posts para treino, visto que ficou tudo preso no teste (ou quase tudo se o gpt discordar, mas não deixa de ter sido queimado no conjunto de teste que está no json)

(contagens de posts por setor e sentimento para teste e treino atuais em baixo)

posto isso, será boa ideia gerar dados sintéticos:

- gerar um txt por post para os setores que precisam usando o gpt

- passar o post pelos 3 llms para obter o sentimento esperado

- inserir o post e o sentimento esperado no dataset de treino

## contagens

In [10]:
import json
import pandas as pd

with open(str(PROJECT_ROOT / "01_data/processed/reddit_gpt_validation_final.json"), "r") as f:
    reddit_gpt_validation_with_labels = json.load(f)

test_data_per_sector = {}
for post_id, content in reddit_gpt_validation_with_labels.items():
    for sector in content["label"]:
        expected = content["label"][sector]["expected_sentiment"]
        gpt_label = content["label"][sector]["gpt_sentiment"]
        
        if sector not in test_data_per_sector:
            test_data_per_sector[sector] = {}
        if expected not in test_data_per_sector[sector]:
            test_data_per_sector[sector][expected] = 0
        if gpt_label == expected:
            test_data_per_sector[sector][expected] += 1

pd.DataFrame(test_data_per_sector).T

,neutral,positive,negative
Information Technology,807,89,51
Health Care,988,39,28
Materials,942,35,20
Financials,844,40,53
Consumer Discretionary,791,52,53
Utilities,1057,20,9
Communication Services,991,30,20
Real Estate,969,27,42
Consumer Staples,1020,23,17
Energy,949,54,42


In [11]:
raw_data = pd.read_parquet(str(PROJECT_ROOT / "02_sentiment/reddit/dataset.parquet"))

def majority_vote(cell):
    votes = {"neutral": 0, "positive": 0, "negative": 0}
    for label in cell:
        if label in votes:
            votes[label] += 1
    majority_label = max(votes, key=votes.get)
    
    return majority_label if votes[majority_label] > 1 else "neutral"

train_data = raw_data.copy()
for sector in [sec for sec in train_data.columns if sec.startswith("label_")]:
    train_data[sector] = train_data[sector].apply(majority_vote)

train_data = train_data[~train_data.index.isin(set(reddit_gpt_validation_with_labels.keys()))]

train_data_per_sector = {}
for sector in [sec for sec in train_data.columns if sec.startswith("label_")]:
    train_data_per_sector[sector] = dict(train_data[sector].value_counts())

pd.DataFrame(train_data_per_sector).T.fillna(0).astype(int)

,neutral,positive,negative
label_Information Technology,16282,1129,456
label_Health Care,17470,319,78
label_Materials,17727,99,41
label_Financials,16904,199,764
label_Consumer Discretionary,16650,769,448
label_Utilities,17867,0,0
label_Communication Services,17786,51,30
label_Real Estate,17728,91,48
label_Consumer Staples,17837,26,4
label_Energy,17470,255,142


## dados sintéticos

In [12]:
# - gerar um txt por post para os setores que precisam usando o gpt

# - passar o post pelos 3 llms para obter o sentimento esperado

# - inserir o post e o sentimento esperado no dataset de treino

# BIG BIRD VS MODERN BERT : tlvz modern bert ganhe?!

In [13]:
import os
import json

def create_text(content):

    content = json.loads(content)
    text = f"""[POST]
Title:
{content.get("title", "<insert title>")}
Body:
{content.get("body", "<insert body>")}\n
[COMMENTS]
"""
    comments = content.get("comments", [])
    for idx, comment in enumerate(comments):
        text += f"Comment {idx+1}:\n{comment}\n"
    
    if not comments:
        text += "No comments available.\n"
    
    return text

content = True
done_count = 0
# for content in aaaaaa:
while content:
    content = input("Enter the content of the post (or 'exit' to finish):")
    if content.lower() == "exit":
        break

    # get higherst existing file number
    existing_files = [f for f in os.listdir("synthetic_posts2") if f.startswith("sy2") and f.endswith(".txt")]
    existing_numbers = [int(f[3:6]) for f in existing_files if f[3:6].isdigit()]
    next_number = max(existing_numbers) + 1 if existing_numbers else 1

    file_name = f"sy2{next_number:03d}.txt"
    file_path = os.path.join("synthetic_posts2", file_name)

    with open(file_path, "w") as f:
        f.write(create_text(content))
    
    done_count += 1

    print(f"you already did {0 - done_count} in this go! ;number {next_number} saved")

30 bear consumer staples

In [14]:
# create a reddit post that would be in a subreddit such as stocks, investing or wallstreetbets that follow the format:

# {
#     title: <title of the post>,
#     "body": <body of the post>,
#     "comments": [
#         <comment 1>,
#         <comment 2>,
#         <comment 3>,
#         ...
#     ]
# }

# it should be related and bull for the utilities sector, without mentioning specific bulisshined. like a rational. also feel free to vary the number of comments, type of languange (formal, informal, etc.) to create a more diverse dataset. note that the objective is to augmentente a dataset for sentiment classification for the gics sectors, so the more diverse the better.

In [15]:
import os


# syntethic_posts = {}

# for filename in [filename for filename in os.listdir("synthetic_posts") if filename.endswith(".txt")]:
#     with open(os.path.join("synthetic_posts", filename), "r") as f:
#         content = f.read()
    
#     syntethic_posts[filename.replace(".txt", "")] = content

# syntethic_posts = pd.DataFrame.from_dict(syntethic_posts, orient="index", columns=["content"])
# syntethic_posts.sort_index(inplace=True)

# syntethic_posts = syntethic_posts.loc["syd810":]


syntethic_posts2 = {}
for filename in [filename for filename in os.listdir("synthetic_posts2") if filename.endswith(".txt")]:
    with open(os.path.join("synthetic_posts2", filename), "r") as f:
        content = f.read()
    
    syntethic_posts2[filename.replace(".txt", "")] = content

syntethic_posts2 = pd.DataFrame.from_dict(syntethic_posts2, orient="index", columns=["content"])
syntethic_posts2.sort_index(inplace=True)


# merged_posts = pd.concat([syntethic_posts, syntethic_posts2])
merged_posts = syntethic_posts2.loc["sy2101":]


merged_posts.to_parquet("synthetic_posts.parquet")

merged_posts.shape, merged_posts.head(), merged_posts.tail()

((94, 1),
                                                   content
 sy2101  [POST]\nTitle:\nWhy is nobody talking about th...
 sy2102  [POST]\nTitle:\nUtilities: the ultimate stealt...
 sy2103  [POST]\nTitle:\nWallStreetBets will hate this:...
 sy2104  [POST]\nTitle:\nSerious question: are utilitie...
 sy2105  [POST]\nTitle:\nMy boomer portfolio is outperf...,
                                                   content
 sy2190  [POST]\nTitle:\nStaples face talent competitio...
 sy2191  [POST]\nTitle:\nInventory mismanagement risks ...
 sy2192  [POST]\nTitle:\nHealth regulations could resha...
 sy2193  [POST]\nTitle:\nStaples’ defensive reputation ...
 sy2194  [POST]\nTitle:\nOpportunity cost: tying up cap...)

In [16]:
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!
# # https://www.kaggle.com/code/hugoverssimo/notebook6aa89b3c10 # REQUIRES UPDATE THE DATASET with the new synthetic data !!!!

# import pyperclip
# import json

# message = (
#     "You are a financial-domain impact classifier for equity sectors. Your task is to analyze a Reddit post and classify the expected impact of the described events on specific GICS sectors\n."
#     "Classification Rules:\n"
#     '- "positive": applies only if the thread expresses MEANINGFUL bullishness, optimism, expected gains, or favorable news for a sector.\n'
#     '- "negative": applies only if the thread expresses MEANINGFUL bearishness, pessimism, expected declines, or harmful news for a sector.\n'
#     '- "neutral": applies when the thread does not meaningfully reference the sector, sentiment is unclear, or discussion is casual/speculative without substance.\n'
#     "\n"
#     'Definition of "meaningful":\n'
#     "- A clear expectation of price movement or performance (up or down)\n"
#     "- A stated reason, catalyst, or justification (e.g., earnings, news, macro factors, valuation)\n"
#     "- Repeated or emphasized sentiment across multiple comments\n"
#     "- Explicit long or short positioning with intent\n"
#     "\n"
#     'The following are NOT meaningful by themselves:\n'
#     "- Casual mentions of sectors without sentiment (e.g., answering what lists of stocks, and stuff like that)\n"
#     "- Jokes, memes, sarcasm, or one-line opinions\n"
#     "- Watchlists, tickers without commentary, or gap lists\n"
#     '- Statements without reasoning (e.g., "this stock will moon" or "short it" without explanation)\n'
#     "\n"
#     "Output Requirements: Return a single JSON object with the sentiment for each sector mentioned in the post. The JSON should have the format:\n"
#     "{\n"
#     '   "Communication Services": "<positive | negative | neutral>",\n'
#     '   "Consumer Discretionary": "<positive | negative | neutral>",\n'
#     '   "Consumer Staples": "<positive | negative | neutral>",\n'
#     '   "Energy": "<positive | negative | neutral>",\n'
#     '   "Financials": "<positive | negative | neutral>",\n'
#     '   "Health Care": "<positive | negative | neutral>",\n'
#     '   "Industrials": "<positive | negative | neutral>",\n'
#     '   "Information Technology": "<positive | negative | neutral>",\n'
#     '   "Materials": "<positive | negative | neutral>",\n'
#     '   "Real Estate": "<positive | negative | neutral>",\n'
#     '   "Utilities": "<positive | negative | neutral>"\n'
#     "}\n"
#     "Classify strictly according to these rules. Output only the JSON object, nothing else. Are you ready?"
# )

# pyperclip.copy(message)
# ready = input("Message copied to clipboard. Are you ready to classify the Reddit posts?")



# raw_labels = pd.read_parquet("reddit_synthetic_labeled_sy2194.parquet")

# sectors = [col.replace("label_", "") for col in train_data.columns if col.startswith("label_")]

# new_parquet = {}

# for i, row in raw_labels.iterrows():

#     llama = row["meta-llama/Llama-3.1-8B-Instruct"]
#     gemma = row["google/gemma-7b-it"]
#     qewn = row["Qwen/Qwen2.5-7B-Instruct"]

#     if llama.get("error", None) or qewn.get("error", None):
#         continue

#     if gemma.get("error", None):
#         # replace with gpt

#         try:
#             with open(f"synthetic_posts/{i}.txt", "r") as f:
#                 thread = f.read()
#         except:
#             with open(f"synthetic_posts2/{i}.txt", "r") as f:
#                 thread = f.read()
        
#         pyperclip.copy(thread)

#         new_entry = input("new entry for gemma:")

#         try:
#             json.loads(new_entry)
#         except json.JSONDecodeError:
#             print("Invalid JSON. Skipping this entry.")
#             continue
        
#         gemma = json.loads(new_entry)
       

#     # save into the cell
#     new_parquet[i] = {
#         "meta-llama/Llama-3.1-8B-Instruct": llama,
#         "google/gemma-7b-it": gemma,
#         "Qwen/Qwen2.5-7B-Instruct": qewn
#     }

#     print(f"Processed {i}")

# new_parquet_df = pd.DataFrame.from_dict(new_parquet, orient="index")
# new_parquet_df.to_parquet("reddit_synthetic_labeled_XXX.parquet")

In [17]:
sectors = [col.replace("label_", "") for col in train_data.columns if col.startswith("label_")]

sydXXX_labeled = [
    "reddit_synthetic_labeled_106.parquet",
    "reddit_synthetic_labeled_247.parquet",
    "reddit_synthetic_labeled_444.parquet",
    "reddit_synthetic_labeled_809.parquet",
    "reddit_synthetic_labeled_999.parquet",
    "reddit_synthetic_labeled_194.parquet",
    ]
sydXXX_labeled = pd.concat([pd.read_parquet(f) for f in sydXXX_labeled])

new_parquet_df_for_concat = {}

for i, row in sydXXX_labeled.iterrows():
    skip_post = False

    llama = row["meta-llama/Llama-3.1-8B-Instruct"]
    gemma = row["google/gemma-7b-it"]
    qewn = row["Qwen/Qwen2.5-7B-Instruct"]

    sector_votes = {}
    for sector in sectors:
        votes = {"positive": 0, "neutral": 0, "negative": 0}

        try:
            votes[llama[sector]] += 1
            votes[gemma[sector]] += 1
            votes[qewn[sector]] += 1
        except KeyError:
            skip_post = True
            break
    
        majority_label = max(votes, key=votes.get) if votes[max(votes, key=votes.get)] > 1 else "neutral"

        sector_votes[f"label_{sector}"] = majority_label
    
    try:
        with open(f"synthetic_posts/{i}.txt", "r") as f:
            thread = f.read()
    except:
        with open(f"synthetic_posts2/{i}.txt", "r") as f:
            thread = f.read()


    sector_votes["thread"] = thread

    if not skip_post:
        new_parquet_df_for_concat[i] = sector_votes

a_new_df = pd.DataFrame.from_dict(new_parquet_df_for_concat, orient="index")

train_and_synthetic = pd.concat([train_data, a_new_df], ignore_index=False)

train_data_per_sector = {}
for sector in [sec for sec in train_and_synthetic.columns if sec.startswith("label_")]:
    train_data_per_sector[sector] = dict(train_and_synthetic[sector].value_counts())

pd.DataFrame(train_data_per_sector).T.fillna(0).astype(int)

,neutral,positive,negative
label_Information Technology,17097,1190,511
label_Health Care,18374,323,101
label_Materials,18582,111,105
label_Financials,17779,203,816
label_Consumer Discretionary,17442,807,549
label_Utilities,18588,103,107
label_Communication Services,18574,115,109
label_Real Estate,18584,106,108
label_Consumer Staples,18578,113,107
label_Energy,18338,287,173


In [19]:
a_new_df.to_parquet("all_synthetic_labeled.parquet")